<a href="https://colab.research.google.com/github/Fisher12quest/Datos-student-performance/blob/main/Ejercicio1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Paso 1: Configuracion entorno - Instalacion de librerias

In [21]:
import pandas as pd
import numpy as np
import polars as pl

Paso 2: Generar base de datos sintetica

In [22]:
# Definir una base de datos con 5 millones de registros
n_rows = 5_000_000
np.random.seed(42)

print("Generando datos sintéticos...")

# Generar datos de manera eficiente usando numpy
id_transaccion = np.arange(1, n_rows + 1)

# Fechas aleatorias en un rango determinado
fecha_inicio = np.datetime64('2023-01-01')
fecha_fin = np.datetime64('2023-12-31')
dias_rango = (fecha_fin - fecha_inicio).astype(int)
fecha = fecha_inicio + np.random.randint(0, dias_rango, size=n_rows).astype('timedelta64[D]')

# Categorías y variables cualitativas
regiones = ['Norte', 'Sur', 'Este', 'Oeste', 'Centro']
categorias = ['Electrónica', 'Ropa', 'Hogar', 'Alimentos', 'Deportes']
sucursales = [f'Sucursal_{i}' for i in range(1, 21)]

region = np.random.choice(regiones, size=n_rows)
categoria = np.random.choice(categorias, size=n_rows)
sucursal = np.random.choice(sucursales, size=n_rows)

# Montos aleatorios (por ejemplo, entre 5.0 y 1000.0)
monto = np.round(np.random.uniform(5.0, 1000.0, size=n_rows), 2)

# Crear el DataFrame de Pandas
df = pd.DataFrame({
    'id_transaccion': id_transaccion,
    'fecha': fecha,
    'region': region,
    'categoria': categoria,
    'sucursal': sucursal,
    'monto': monto
})

print("¡Base de datos generada con éxito!")
display(df.head())

Generando datos sintéticos...
¡Base de datos generada con éxito!


,id_transaccion,fecha,region,categoria,sucursal,monto
0,1,2023-04-13,Sur,Ropa,Sucursal_20,906.24
1,2,2023-12-15,Este,Alimentos,Sucursal_8,135.46
2,3,2023-09-28,Este,Alimentos,Sucursal_16,472.29
3,4,2023-04-17,Oeste,Electrónica,Sucursal_7,165.21
4,5,2023-03-13,Norte,Alimentos,Sucursal_13,770.96


Paso 3: Transformacion con Pandas

In [23]:
df_pandas = df.copy()

In [24]:
%%time
# Lógica de pandas corregida

# Agrupamos por 'region' y 'categoria' para sumar el 'monto'
resultado_pd = (
    df_pandas.groupby(['region', 'categoria'])['monto']
    .sum()
    .reset_index()
)

# Mostrar el resultado
display(resultado_pd)

,region,categoria,monto
0,Centro,Alimentos,1.007405e+08
1,Centro,Deportes,9.992975e+07
2,Centro,Electrónica,1.004978e+08
3,Centro,Hogar,1.005503e+08
4,Centro,Ropa,1.006698e+08
5,Este,Alimentos,1.005390e+08
6,Este,Deportes,1.004563e+08
7,Este,Electrónica,1.006162e+08
8,Este,Hogar,1.007813e+08
9,Este,Ropa,1.001130e+08


CPU times: user 1.5 s, sys: 236 ms, total: 1.74 s
Wall time: 1.78 s


Paso 4: Transformación con Polars

In [25]:
%%time
# Convertir nuestro DataFrame de Pandas a Polars
df_polars = pl.from_pandas(df_pandas)

# Lógica equivalente en Polars
resultado_pl = (
    df_polars.group_by(['region', 'categoria'])
    .agg(pl.col('monto').sum())
)

# Mostrar el resultado
display(resultado_pl.head())

region,categoria,monto
str,str,f64
"""Oeste""","""Electrónica""",1.0034e8
"""Norte""","""Alimentos""",1.0037e8
"""Este""","""Deportes""",1.0046e8
"""Norte""","""Deportes""",1.0068e8
"""Sur""","""Deportes""",1.0013e8


CPU times: user 2.02 s, sys: 173 ms, total: 2.19 s
Wall time: 2.03 s


In [26]:
#logica en Polars
df_polars = pl.DataFrame(df_pandas)

In [27]:
%%time
# Lógica de Polars corregida

# Agrupamos por 'region' y 'categoria' para sumar el 'monto'
resultado_pl = (
    df_polars.group_by(['region', 'categoria'])
    .agg(pl.col('monto').sum())
)

# Mostrar el resultado
display(resultado_pl)

region,categoria,monto
str,str,f64
"""Este""","""Alimentos""",1.0054e8
"""Oeste""","""Electrónica""",1.0034e8
"""Norte""","""Alimentos""",1.0037e8
"""Este""","""Deportes""",1.0046e8
"""Norte""","""Deportes""",1.0068e8
…,…,…
"""Sur""","""Electrónica""",1.0043e8
"""Norte""","""Hogar""",1.0017e8
"""Oeste""","""Ropa""",1.0047e8


CPU times: user 393 ms, sys: 1.75 ms, total: 395 ms
Wall time: 222 ms
